In [1]:
# Importacoees e Inicializacao da SparkSession
from pyspark.sql import SparkSession

# Inicializa a sessao utilizando parenteses para evitar erros de indentacao
# Os pacotes JAR para conexao com Postgres e MinIO sao injetados dinamicamente
spark = (SparkSession.builder
    .appName("Formatos-Spark-CSV")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-avro_2.12:3.5.0")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "root-minio")
    .config("spark.hadoop.fs.s3a.secret.key", "root12345678")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print(f"SparkSession ativa. Versao: {spark.version}")

SparkSession ativa. Versao: 3.5.0


In [4]:
# Extracao (Extract) do PostgreSQL via JDBC
print("Conectando ao PostgreSQL via JDBC e extraindo dados financeiros...")

df_financeiro = (spark.read.jdbc(
    url="jdbc:postgresql://postgres:5432/postgres",
    table="transacoes_financeiras",
    properties={"user": "postgres", "password": "tailwind2026", "driver": "org.postgresql.Driver"}
))

# Exibe o esquema mapeado pelo driver JDBC
df_financeiro.printSchema()

Conectando ao PostgreSQL via JDBC e extraindo dados financeiros...
root
 |-- id_transacao: string (nullable = true)
 |-- conta_origem: string (nullable = true)
 |-- conta_destino: string (nullable = true)
 |-- tipo_movimento: string (nullable = true)
 |-- valor: decimal(15,4) (nullable = true)
 |-- moeda: string (nullable = true)
 |-- data_transacao: timestamp (nullable = true)
 |-- hash_auditoria: string (nullable = true)
 |-- data_criacao: timestamp (nullable = true)



In [7]:
# Carga (Load) para o MinIO em formato CSV
bucket_name = "raw"
destino_s3 = f"s3a://{bucket_name}/spark_jobs/transacoes_csv/"

print(f"Gravando dados em formato CSV no Object Storage: {destino_s3}")

# Gravacao configurada com inclusao de cabecalho e delimitador ponto e virgula
(df_financeiro.write
    .mode("overwrite")
    .option("header", "true")
    .option("sep", ";")
    .csv(destino_s3)
)

print("Escrita do CSV concluida com sucesso no destino.")

Gravando dados em formato CSV no Object Storage: s3a://raw/spark_jobs/transacoes_csv/
Escrita do CSV concluida com sucesso no destino.


In [8]:
# Auditoria e Validação dos Dados Gravados
print("Lendo arquivos CSV de volta do Data Lake para validacao...")

df_validacao = (spark.read
    .option("header", "true")
    .option("sep", ";")
    .csv(destino_s3)
)

print(f"Quantidade total de registros localizados: {df_validacao.count()}")
df_validacao.show(5, truncate=False)

Lendo arquivos CSV de volta do Data Lake para validacao...
Quantidade total de registros localizados: 20000
+------------------------------------+------------------+------------------+--------------+----------+-----+------------------------+----------------------------------------------------------------+------------------------+
|id_transacao                        |conta_origem      |conta_destino     |tipo_movimento|valor     |moeda|data_transacao          |hash_auditoria                                                  |data_criacao            |
+------------------------------------+------------------+------------------+--------------+----------+-----+------------------------+----------------------------------------------------------------+------------------------+
|e9b4ba43-ef07-4d76-8436-4c829966c49b|IYXG33090884540679|NMIZ81024723375965|DEBITO        |14236.8874|BRL  |2026-04-16T23:53:15.000Z|e23c1a3adb95d63fdc32c8eedd0f87db6fd050073b12e7359e4af8d17677ab40|2026-04-16T23:53:15.00

In [9]:
# Encerramento da Sessão e Limpeza de Recursos
print("Encerrando a SparkSession e liberando os recursos da JVM...")
spark.stop()
print("Sessao fechada com seguranca.")

Encerrando a SparkSession e liberando os recursos da JVM...
Sessao fechada com seguranca.


### Análise de Arquitetura: O Formato CSV no Apache Spark

**Características Principais:** Formato baseado em texto plano, estritamente orientado a linhas.

#### Vantagens
* **Universalidade:** É o formato mais democrático do mercado. Qualquer ferramenta, desde o Microsoft Excel até sistemas legados em C# ou COBOL, consegue ler um arquivo CSV sem necessidade de bibliotecas complexas.
* **Transparência:** Por ser texto puro, pode ser aberto e inspecionado visualmente em qualquer editor de blocos de notas para depuração rápida.

#### Desvantagens
* **Overhead de Processamento:** O Spark precisa converter tipos estruturados (como inteiros, decimais e timestamps) em strings textuais na escrita, e fazer o processo inverso (inferência de schema) na leitura, o que consome CPU.
* **Ausência de Tipagem Nativa:** O CSV não armazena metadados de esquema. Dados numéricos de alta precisão perdem o rigor técnico se não forem remapeados manualmente na leitura.
* **Impacto do Paralelismo:** Por ser uma engine distribuída, o Spark criará uma pasta e salvará múltiplos arquivos parciais (como part-00000, part-00001) dependendo de como os dados estavam distribuídos no cluster. Isso pode confundir usuários de negócios que esperam um arquivo único.

#### Casos de Uso no Mercado
* Compartilhamento final de dados agregados e tratados com equipes de negócio ou parceiros externos.
* Exportação de tabelas de parametrização estáticas e de baixíssimo volume (tabelas de De-Para).